# LLM-based morphological annotation corrections

EstNLTK has tools for combining Vabamorf with LLM-s for getting corrections for morphological analyses [(Saska 2026)](https://thesis.cs.ut.ee/8f10695d-4276-47ea-b306-5df66c034057). 

## Prerequisites

*Note: you need to install [estnltk_neural](https://github.com/estnltk/estnltk/tree/main/estnltk_neural) and [openai](https://pypi.org/project/openai) packages for using the LLM-based morphological corrections.*

First, you need to set up an LLMProvider that mediates communication with the LLM. 
Currently, `AzureOpenAIProvider` is available.

In [1]:
from estnltk_neural.common import AzureOpenAIProvider

Upon initializing AzureOpenAIProvider, provide the following connection parameters:
* `deployment` -- Azure deployment name, which is what the API calls a model;
* `endpoint` -- URL of the Azure endpoint where requests will be sent;
* `api_key` -- Key required for making the API calls;
* `api_version` -- Azure API version name;

Note: you can put `endpoint` into the environment variable ``AZURE_ENDPOINT``, and `api_key` into the environment variable ``OPENAI_API_KEY`` -- in that case, these parameters will be retrieved automatically from the environment variables and you do not need to set them manually. This is also done in the following example.

In [2]:
# Checks whether the api key and endpoint are available in the environment variables
# Comment this out if you want to set these parameters manually in the code
import os
api_key = os.getenv("OPENAI_API_KEY")
assert isinstance(api_key, str), '(!) Missing OpenAI api key. Please set the environment variable OPENAI_API_KEY.'
api_endpoint = os.getenv("AZURE_ENDPOINT")
assert isinstance(api_endpoint, str), '(!) Missing OpenAI endpoint. Please set the environment variable AZURE_ENDPOINT.'

In [3]:
# Initialize LLM via Azure OpenAI API
llm_provider = AzureOpenAIProvider(deployment='EstNLTK-gpt-4o', api_version="2025-01-01-preview")

Other parameters:
* `model_name` -- Name of the model. This is used in the costs estimation as the defined by the parameter `costs` below;
* `requests_per_minute` -- Used to space requests out. The default matches the rate limit the experiments ran against.
* `max_attempts` -- Attempts per request, with exponential backoff between them.
* `max_cost_usd` --  Stop the run once the estimated spend passes this. Note: LLMProvider does not automatically retrieve up-to-date pricing information, you should manually configure prices via the parameter `costs`. ``None`` disables the ceiling, which is reasonable for interactive prompting and not for a corpus processing. 
* `costs` -- Pricing information: a dictionary mapping from model name to input and output costs (in USD). This will be used for cost estimation and max cost checking. Defaults to: `{"gpt-4o": {"input": 2.50 / 1_000_000, "output": 10.00 / 1_000_000}}` (input: 2.5 USD per 1000000 tokens, output: 10.00 USD per 1000000 tokens). For up-to-date pricing information, please see: https://azure.microsoft.com/en-us/pricing/details/azure-openai

## LLMWordReplacementGenerator

`LLMWordReplacementGenerator` asks a language model to propose word forms that could replace the word at a given position. You can use the parameter `system_prompt` to change the prompt that will be used when communicating with the LLM.

Usage example:

In [4]:
from estnltk_neural.taggers import LLMWordReplacementGenerator
word_replacement_generator = LLMWordReplacementGenerator(provider=llm_provider, n_candidates=10)

In [5]:
# Default prompt
print( word_replacement_generator.system_prompt )

You are an Estonian sentence rewriting assistant.

Replace exactly one marked token in angle brackets <...> with a context-appropriate alternative.

Rules:
- Preserve tense, number, case, agreement, capitalisation, punctuation, and word order as much as possible.
- Infer the token's grammatical role and morphology from the sentence context.
- Generate 10 alternatives that fit the context and use only these cases: nominative, genitive, partitive, or additive/illative (both fit the same role in this task).
- If the token is a proper name, replace it with another plausible proper name that still fits the required case.
- Do not repeat the original token unless it is the only valid option.



In [6]:
# Ask for replacements for a word with index 2
replacements = word_replacement_generator.generate(['Mina', 'söön', 'muna', '.'], 2)
replacements

[('õuna', None),
 ('leiba', None),
 ('pirni', None),
 ('saiakest', None),
 ('puuvilja', None),
 ('banaani', None),
 ('kiivit', None),
 ('kartulit', None),
 ('tomatit', None),
 ('muna', None)]

## LLMWordFilter

`LLMWordFilter` asks a language model which of the given candidates fit into the sentence at the given position. You can use the parameter `system_prompt` to change the prompt that will be used when communicating with the LLM.

Usage example:

In [7]:
from estnltk_neural.taggers import LLMWordFilter
word_filter = LLMWordFilter(provider=llm_provider, keep_n=5)

In [8]:
# Default prompt
print( word_filter.system_prompt )

You are an Estonian sentence rewriting assistant.
Your task is to select the best candidates from the provided list to replace the marked word <...> in the sentence.

Rules:
- Do not invent new candidates.
- Preserve the word's tense, number, case agreement, punctuation, and capitalisation.
- Return candidates that fit naturally and grammatically in one of these cases: nominative, genitive, partitive, or additive/illative.
- For proper names, keep only proper-name candidates that fit the same grammatical context.
- If fewer valid candidates exist than requested, return as many as possible.
- Order candidates from best to worst fit.
- Do not repeat the original word unless it is the only valid option.



In [9]:
replacements

[('õuna', None),
 ('leiba', None),
 ('pirni', None),
 ('saiakest', None),
 ('puuvilja', None),
 ('banaani', None),
 ('kiivit', None),
 ('kartulit', None),
 ('tomatit', None),
 ('muna', None)]

In [10]:
# Ask which of the given replacements for a word with index 2 should be kept
word_filter.filter( ['Mina', 'söön', 'muna', '.'], 2, replacements )

[('õuna', None),
 ('leiba', None),
 ('pirni', None),
 ('saiakest', None),
 ('puuvilja', None)]

## WordReplacementAnalyzer

`WordReplacementAnalyzer` uses `VabamorfAnalyzer` to get a distribution over morphological forms from candidate replacements provided by `LLMWordReplacementGenerator` and `LLMWordFilter` (optional).

You should initialize `LLMWordReplacementGenerator` beforehand and pass it to `WordReplacementAnalyzer` to get distribution estimates. Optionally, you can also provide `LLMWordFilter` for filtering the list of candidates.

In [11]:
from estnltk_neural.taggers import WordReplacementAnalyzer
word_replacement_analyzer = WordReplacementAnalyzer( word_replacement_generator, word_filter )

In [12]:
# Ask for the morphological distribution of a word with index 2
word_replacement_analyzer.analyze(['Mina', 'söön', 'muna', '.'], 2)

[('adt', 0.35), ('sg p', 0.35), ('sg g', 0.24999999999999994), ('sg es', 0.05)]

**Important:** `WordReplacementAnalyzer` relies on **estnltk's default tokenization** on providing the morphological analysis. 
If the tokenization of the input sentence differs from the estnltk's default tokenization, then `WordReplacementAnalyzer` may not be able to retrieve morphological analysis for the word with the given index due to the tokenization misalignment.

It is recommended to always tokenize the input sentence via estnltk (e.g. `Text('Mina söön muna.').tag_layer('sentences')['sentences'].text`) to get the suitable tokenization. If you are unable to do so, then set `force_whitespace_tokenization=True` in `WordReplacementAnalyzer` to force whitespace tokenization of the input. However, be aware that the whitespace tokenization does not separate punctuation from words, which may affect morphological analysis results.

## LLMmorphCorrectionsRetagger

`LLMmorphCorrectionsRetagger` corrects selected words of a Vabamorf morphological analysis layer via the replacement analysis described above.

You should initialize `WordReplacementAnalyzer` beforehand and define a function for selecting spans that need to be corrected.

In [13]:
from estnltk_neural.taggers import LLMmorphCorrectionsRetagger

In [14]:
# Define a function for selecting words which will be corrected
def is_target_word( span ):
    return span.text.lower() == 'muna'

In [15]:
replacement_retagger = LLMmorphCorrectionsRetagger( analyzer=word_replacement_analyzer, select=is_target_word )

In [16]:
replacement_retagger

LLMmorphCorrectionsRetagger(analyzer=<estnltk_neural.taggers.neural_morph.llm_generator_based.word_replacement_analyzer.WordReplacementAnalyzer object at 0x000002117130EBA0>, select=<function is_target_word at 0x000002117136F8A0>, flag_attribute=llm_morph_correction, sentences_layer=sentences, words_layer=words, slang_lex=True, compound=True, phonetic=False, stem=False, use_postanalysis=True, candidate_layer_name=morph_analysis__vm_candidates, check_output_consistency=False)

In [17]:
# Define input text
from estnltk import Text
text = Text('Mina söön muna.').tag_layer('morph_analysis')

In [18]:
# Check the default analysis
text['morph_analysis']

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Mina', [{'normalized_text': 'Mina', 'lemma': 'mina', 'root': 'mina', 'root_tokens': ['mina'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P'}]),
Span('söön', [{'normalized_text': 'söön', 'lemma': 'sööma', 'root': 'söö', 'root_tokens': ['söö'], 'ending': 'n', 'clitic': '', 'form': 'n', 'partofspeech': 'V'}]),
Span('muna', [{'normalized_text': 'muna', 'lemma': 'muna', 'root': 'muna', 'root_tokens': ['muna'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

In [19]:
# Apply replacement analysis via LLM
replacement_retagger.retag( text )

Text(text='Mina söön muna.')

In [20]:
# Check results
text['morph_analysis']

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech', 'llm_morph_correction'), spans=SL[Span('Mina', [{'normalized_text': 'Mina', 'lemma': 'mina', 'root': 'mina', 'root_tokens': ['mina'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P', 'llm_morph_correction': 'none'}]),
Span('söön', [{'normalized_text': 'söön', 'lemma': 'sööma', 'root': 'söö', 'root_tokens': ['söö'], 'ending': 'n', 'clitic': '', 'form': 'n', 'partofspeech': 'V', 'llm_morph_correction': 'none'}]),
Span('muna', [{'normalized_text': 'muna', 'lemma': 'muna', 'root': 'muna', 'root_tokens': ['muna'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S', 'llm_morph_correction': 'agreed'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z', 'llm_morph_correction': 'none'}])])

In [21]:
# Finally, check the estimated spending (USD)
llm_provider.spent_usd

0.005540000000000001